# SNGP: Spectral-Normalized Neural Gaussian Process

Single forward pass distance-aware uncertainty.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from deepuq.models import MLP
from deepuq.methods.sngp import SNGPWrapper

## Generate 2-Class Spiral Data

In [ ]:
np.random.seed(42)

def make_spiral(n_points=100, noise=0.2):
    """Generate 2-class spiral dataset."""
    theta = np.linspace(0, 2 * np.pi, n_points)
    r = np.linspace(0.5, 2, n_points)
    
    # Class 0
    x0 = r * np.cos(theta) + noise * np.random.randn(n_points)
    y0 = r * np.sin(theta) + noise * np.random.randn(n_points)
    
    # Class 1 (rotated by pi)
    x1 = r * np.cos(theta + np.pi) + noise * np.random.randn(n_points)
    y1 = r * np.sin(theta + np.pi) + noise * np.random.randn(n_points)
    
    X = np.vstack([np.column_stack([x0, y0]), np.column_stack([x1, y1])])
    y = np.array([0]*n_points + [1]*n_points)
    return X, y

X_train, y_train = make_spiral(100, noise=0.2)
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)

plt.scatter(X_train[:100, 0], X_train[:100, 1], c='blue', alpha=0.6, label='Class 0')
plt.scatter(X_train[100:, 0], X_train[100:, 1], c='red', alpha=0.6, label='Class 1')
plt.legend()
plt.title('2-Class Spiral Dataset')
plt.axis('equal')
plt.show()

## Create and Train SNGP Model

In [ ]:
import torch.nn as nn

# MLP uses nn.Sequential internally with no named last layer.
# Define a custom model with a named 'output' layer for SNGP.
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.hidden = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU()
        )
        self.output = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        return self.output(self.hidden(x))

# Create base model and wrap with SNGP
base_model = SimpleMLP(input_dim=2, hidden_dim=64, output_dim=2)
sngp_model = SNGPWrapper(base_model, last_layer_name='output', num_random_features=512, spec_norm_bound=6.0)

optimizer = torch.optim.Adam(sngp_model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

# Training loop
sngp_model.train()
for epoch in range(100):
    # Reset covariance at start of epoch
    sngp_model.reset_covariance()
    
    optimizer.zero_grad()
    logits = sngp_model(X_train_t)
    loss = loss_fn(logits, y_train_t)
    loss.backward()
    optimizer.step()
    
    # Update covariance with training data
    sngp_model.update_covariance(X_train_t)
    
    if (epoch + 1) % 25 == 0:
        acc = (logits.argmax(dim=1) == y_train_t).float().mean()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Acc: {acc.item():.3f}")

## Predict with Uncertainty

In [ ]:
# Create grid of test points
xx = np.linspace(-4, 4, 80)
yy = np.linspace(-4, 4, 80)
XX, YY = np.meshgrid(xx, yy)
grid_points = torch.FloatTensor(np.column_stack([XX.ravel(), YY.ravel()]))

# Predict
sngp_model.eval()
result = sngp_model.predict_uq(grid_points)

probs = result.probs.numpy()  # (N, 2)
epistemic_var = result.epistemic_var.numpy()  # (N, 2)

print(f"Prediction shape: {probs.shape}")
print(f"Uncertainty shape: {epistemic_var.shape}")
print(f"Method: {result.metadata['method']}")

## Visualize Decision Boundary with Uncertainty

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Decision boundary (class 1 probability)
prob_grid = probs[:, 1].reshape(80, 80)
ax = axes[0]
c = ax.contourf(XX, YY, prob_grid, levels=20, cmap='RdBu_r', alpha=0.8)
ax.scatter(X_train[:100, 0], X_train[:100, 1], c='blue', edgecolors='k', s=20)
ax.scatter(X_train[100:, 0], X_train[100:, 1], c='red', edgecolors='k', s=20)
ax.set_title('SNGP Class Probabilities')
plt.colorbar(c, ax=ax)

# Epistemic uncertainty (mean across classes)
unc_grid = epistemic_var.mean(axis=1).reshape(80, 80)
ax = axes[1]
c = ax.contourf(XX, YY, unc_grid, levels=20, cmap='viridis', alpha=0.8)
ax.scatter(X_train[:100, 0], X_train[:100, 1], c='blue', edgecolors='k', s=20)
ax.scatter(X_train[100:, 0], X_train[100:, 1], c='red', edgecolors='k', s=20)
ax.set_title('SNGP Epistemic Uncertainty (higher = more uncertain)')
plt.colorbar(c, ax=ax, label='Epistemic variance')

plt.tight_layout()
plt.show()

print("Note: Uncertainty is higher far from training data, demonstrating")
print("SNGP's distance-aware uncertainty estimation.")